In [1]:
import numpy as np
import sys

# NumPyがどこから読み込まれているか、パスを直接出力
print(f"NumPy Version: {np.__version__}")
print(f"NumPy Path: {np.__file__}")

# もしここで 1.26より上が出たら、Jupyterのカーネル接続先が狂っています
import numba
print("Numba loaded successfully!")

NumPy Version: 1.24.4
NumPy Path: /home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/site-packages/numpy/__init__.py
Numba loaded successfully!


In [3]:
import pandas as pd
import numpy as np
import shap
import os
import random
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

# --- [1] パス・シード設定 ---
# ターゲット: FF / タイプ: n / ファイル: n_all_FP_rebuilt.csv / アルゴリズム: kNN (k=1)
file_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/n_all_FP_rebuilt.csv"
out_dir = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/kNN_SHAP_FF_n_all"
os.makedirs(out_dir, exist_ok=True)

# 再現性のための完全シード固定
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# --- [2] データ読み込みと前処理 ---
print(f"Loading data: {os.path.basename(file_path)}...")
# index_col=0 により1列目のIDを特徴量から確実に除外
df = pd.read_csv(file_path, index_col=0)
df_num = df.select_dtypes(include=[np.number]).dropna()

target = "FF"

# 除外設定の厳格な定義 (博士論文基準を維持)
TARGET_CANDIDATES = ["PCE", "PCEmax", "PCEavg", "Jsc", "Voc", "FF"]
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

# 1. ターゲット候補、不適切変数、および ID由来の "X" 列を確実に除外
features = [c for c in df_num.columns if c not in TARGET_CANDIDATES and c not in INAPPROPRIATE_VARS and c != "X"]

# 2. 物理データパターンの除外
features = [f for f in features if not any(pat in f for pat in PHYSICAL_EXCLUDE_PATTERNS)]

# 3. SD（標準偏差）が 0 の列を完全に排除
features = [f for f in features if df_num[f].std() > 0]

print(f"Final Features count for {target} (n_all_FP): {len(features)}")

X_raw = df_num[features]
y = df_num[target]

# スケーリング（-1 to 1）: kNNは距離計算に基づくため必須
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features, index=X_raw.index)

# --- [3] kNN 学習 ---
print(f"Training kNN (k=1) for {target}...")
# ご指示通り ハイパーパラメータ k=1 で固定
model = KNeighborsRegressor(n_neighbors=1)
model.fit(X_scaled, y)

# モデル精度評価 (全データ学習に対するスコア)
y_pred = model.predict(X_scaled)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("-" * 35)
print(f"Model Performance ({target}):")
print(f"  R2 Score: {r2:.4f}")
print(f"  RMSE:     {rmse:.4f}")
print("-" * 35)

# --- [4] SHAP計算 (KernelExplainer) ---
print(f"Calculating SHAP values for {target} (KernelExplainer)...")
# 背景データのサンプリングを固定 (random_state=SEED)
X_summary = shap.sample(X_scaled, 10, random_state=SEED)
explainer = shap.KernelExplainer(model.predict, X_summary)

# l1_reg=False により、数値の不安定化を防止
shap_values = explainer.shap_values(X_scaled, l1_reg=False)

# --- [5] 保存と可視化 ---
# 1. 重要度CSVの保存
importance_df = pd.DataFrame({
    "Feature": features,
    "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0)
}).sort_values("SHAP_MeanAbs", ascending=False)

out_csv = os.path.join(out_dir, f"{target}_n_all_FP_kNN_SHAP_k1.csv")
importance_df.to_csv(out_csv, index=False)

# 2. 精度サマリーの保存
with open(os.path.join(out_dir, f"{target}_Performance_Summary_k1.txt"), "w") as f:
    f.write(f"Target: {target}\nFile: {os.path.basename(file_path)}\n")
    f.write(f"Hyperparameters: k=1 (Fixed)\n")
    f.write(f"R2 Score: {r2:.4f}\nRMSE: {rmse:.4f}\n")
    f.write(f"Features used: {len(features)}\n")

# 3. SHAP Beeswarm Plot の保存
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_scaled, show=False)
plt.title(f"SHAP Summary: {target} (kNN k=1 - R2={r2:.3f})")
plt.savefig(os.path.join(out_dir, f"SHAP_Beeswarm_{target}_n_all_FP_kNN_k1.png"), bbox_inches='tight', dpi=300)
plt.close()

print(f"Success! All results for {target} (k=1) saved in: {out_dir}")

Loading data: n_all_FP_rebuilt.csv...
Final Features count for FF (n_all_FP): 90
Training kNN (k=1) for FF...
-----------------------------------
Model Performance (FF):
  R2 Score: 0.8452
  RMSE:     0.0327
-----------------------------------
Calculating SHAP values for FF (KernelExplainer)...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 72/72 [00:23<00:00,  3.04it/s]
No data for colormapping provided via 'c'. Parameters 'vmin', 'vmax' will be ignored


Success! All results for FF (k=1) saved in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/kNN_SHAP_FF_n_all
